### 將jupter 處理成可以讀取資料夾下的其他模組

In [1]:
import sys
import os
import pandas as pd
# notebook 所在目錄（通常 cwd = rebuild_data）加入 sys.path
_here = os.path.abspath(os.getcwd())
if _here not in sys.path:
    sys.path.insert(0, _here)


### 引入通用的處理模塊

In [2]:
from parse import read_tej_txt, one_txt, read_tej_folder
import polars as pl


In [3]:
rebuild_root = "/mnt/c/Users/Ultra7/TEJ_data/rebuild_20150601~20260531" 

## 資料 import 
### 引入日歷 
處理成 只有日期 並修正格式

In [4]:
import Calendar

calendar_path = os.path.join(rebuild_root, "calendar", "calendar.txt")
if not os.path.exists(calendar_path):
    raise FileNotFoundError(f"檔案不存在：{calendar_path}，請確認路徑是否正確。")
calendar_df = read_tej_txt(calendar_path, Calendar.CALENDAR_SCHEMA).select(Calendar.keep_col)
calendar_df.head()


date
date
2015-06-01
2015-06-02
2015-06-03
2015-06-04
2015-06-05


### 引入停止交易的日曆

In [5]:
import Trading_Halt

stop_trading_path = os.path.join(rebuild_root, "trading_halt")
trading_halt_df = read_tej_txt(
    one_txt(stop_trading_path),
    Trading_Halt.TRADING_HALT_SCHEMA,
).select(Trading_Halt.keep_col)

# Y9999 為加權指數
# 完全相等 / 部分包含 / 手工 (symbol, 年月日YYYYMMDD)（含任一即 avoid_early=1）
AVOID_EARLY_REASONS: list[str] = ["減資", "現金減資", "違規財報", "自行申請"]
AVOID_EARLY_REASONS_PARTIAL: list[str] = ["面額變更"]
AVOID_EARLY_PAIRS: list[tuple[str, int]] = [
    ("4413", 20180515),  # 前面被打入全額交割 未改善 視為可先預知
    ("2833", 20151002),  # 台灣人壽 併入中國金控 有提前公告
    ("2823", 20211220),  # 中國人壽 併入開發金
]
_exact = pl.col("reason").is_in(AVOID_EARLY_REASONS)
_partial = (
    pl.any_horizontal(
        *[pl.col("reason").str.contains(p, literal=True) for p in AVOID_EARLY_REASONS_PARTIAL]
    )
    if AVOID_EARLY_REASONS_PARTIAL
    else pl.lit(False)
)
_pair_df = (
    pl.DataFrame(AVOID_EARLY_PAIRS, schema={"symbol": pl.Utf8, "_ymd": pl.Int64}, orient="row")
    if AVOID_EARLY_PAIRS
    else pl.DataFrame(schema={"symbol": pl.Utf8, "_ymd": pl.Int64})
).with_columns(pl.lit(True).alias("_manual"))
trading_halt_df = (
    trading_halt_df
    .with_columns(
        pl.col("start_halt").dt.strftime("%Y%m%d").cast(pl.Int64).alias("_ymd")
    )
    .join(_pair_df, on=["symbol", "_ymd"], how="left")
    .with_columns(
        (_exact | _partial | pl.col("_manual").fill_null(False))
        .cast(pl.Int8)
        .alias("avoid_early")
    )
    .drop(["_ymd", "_manual"])
)
trading_halt_df.head()


symbol,start_halt,resume_trading_date,reason,avoid_early
str,date,date,str,i8
"""5466""",2015-06-05,2015-06-17,"""合併下市""",0
"""910708""",2015-06-18,2015-06-22,"""自行申請""",1
"""Y9999""",2015-06-19,null,"""端午節補假""",0
"""Y9999""",2015-07-10,null,"""昌鴻颱風停班""",0
"""6165""",2015-08-10,2015-08-24,"""減資""",1


In [6]:
# schema 已將 start_halt / resume_trading_date 轉成 Date；此格保留作型別確認
assert trading_halt_df["start_halt"].dtype == pl.Date
assert trading_halt_df["resume_trading_date"].dtype == pl.Date
print("trading_halt 日期欄型別 OK")


trading_halt 日期欄型別 OK


In [7]:
# EDA：恢復交易日缺值
resume_empty = pl.col("resume_trading_date").is_null()

print("=== 總覽 ===")
print(f"總筆數: {trading_halt_df.height}")
print(trading_halt_df.select(
    pl.col("start_halt").is_not_null().sum().alias("有開始日"),
    resume_empty.sum().alias("無恢復日"),
    (~resume_empty).sum().alias("有恢復日"),
))

no_resume = trading_halt_df.filter(
    pl.col("start_halt").is_not_null() & pl.col("resume_trading_date").is_null()
)
print(f"\n=== 有開始日、無恢復日：{no_resume.height} 筆 ===")
print("原因分布:")
print(no_resume.group_by("reason").len().sort("len", descending=True))
print(f"\n=== symbol（共 {no_resume['symbol'].n_unique()} 檔）===")
print(no_resume.group_by("symbol").len().sort("len", descending=True))
print("\nsample:")
no_resume.head(20)


=== 總覽 ===
總筆數: 1510
shape: (1, 3)
┌──────────┬──────────┬──────────┐
│ 有開始日 ┆ 無恢復日 ┆ 有恢復日 │
│ ---      ┆ ---      ┆ ---      │
│ u32      ┆ u32      ┆ u32      │
╞══════════╪══════════╪══════════╡
│ 1510     ┆ 217      ┆ 1293     │
└──────────┴──────────┴──────────┘

=== 有開始日、無恢復日：217 筆 ===
原因分布:
shape: (59, 2)
┌──────────────┬─────┐
│ reason       ┆ len │
│ ---          ┆ --- │
│ str          ┆ u32 │
╞══════════════╪═════╡
│ 農曆春節     ┆ 33  │
│ 年前封關     ┆ 20  │
│ 補班不交易   ┆ 17  │
│ 端午節       ┆ 9   │
│ 中秋節       ┆ 9   │
│ …            ┆ …   │
│ 228 彈放     ┆ 1   │
│ 國慶日補假   ┆ 1   │
│ 除夕前一天   ┆ 1   │
│ 端午節補假   ┆ 1   │
│ 昌鴻颱風停班 ┆ 1   │
└──────────────┴─────┘

=== symbol（共 3 檔）===
shape: (3, 2)
┌────────┬─────┐
│ symbol ┆ len │
│ ---    ┆ --- │
│ str    ┆ u32 │
╞════════╪═════╡
│ Y9999  ┆ 215 │
│ 4804   ┆ 1   │
│ 1589   ┆ 1   │
└────────┴─────┘

sample:


symbol,start_halt,resume_trading_date,reason,avoid_early
str,date,date,str,i8
"""Y9999""",2015-06-19,null,"""端午節補假""",0
"""Y9999""",2015-07-10,null,"""昌鴻颱風停班""",0
"""Y9999""",2015-09-28,null,"""中秋節補假""",0
"""Y9999""",2015-09-29,null,"""杜鵑颱風""",0
"""Y9999""",2015-10-09,null,"""國慶日補假""",0
…,…,…,…,…
"""Y9999""",2016-04-05,null,"""清明節""",0
"""Y9999""",2016-05-02,null,"""51彈性放假""",0
"""Y9999""",2016-06-09,null,"""端午節""",0


In [8]:
# 理論上 halt 中 symbol "Y9999" 的日期不會出現在 calendar 中
y9999_halt_dates = trading_halt_df.filter(pl.col("symbol") == "Y9999")["start_halt"]
calendar_dates = calendar_df["date"].unique()
y9999_in_calendar = y9999_halt_dates.filter(y9999_halt_dates.is_in(calendar_dates.to_list()))

if len(y9999_in_calendar) == 0:
    print("理論一致：Y9999 halt 年月日無出現在 calendar 中")
else:
    print("以下 Y9999 halt 年月日 竟出現在 calendar 中，請檢查！")
    print(y9999_in_calendar)


理論一致：Y9999 halt 年月日無出現在 calendar 中


### 處理OHLCV

In [9]:
from OHLCV import OHLCV_SCHEMA

In [10]:
OHLCV_path = os.path.join(rebuild_root, "Daily_Unadjusted_Price")
OHLCV_df = read_tej_folder(OHLCV_path, OHLCV_SCHEMA, allow_market_split=True)
OHLCV_df.head()


symbol,date,open,high,low,close,volume,amount,return_day,turnover,shares_outstanding_thousand,market_value_million,last_bid,last_ask,log_return_day,market_value_pct,trading_value_pct,numtrades,pe_ratio_tse,pe_ratio_tej,pb_ratio_tse,pb_ratio_tej,limit_status,ps_ratio_tej,div_yield_tse,cash_div_yield,price_change,spread,next_open_ref,next_limit_up,next_limit_down,attention_stock,disposed_stock,full_delivery,market
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str
"""1259""",2015-06-01,65.1,65.1,65.1,65.1,1.0,65.0,-0.4587,0.0031,32390.0,2109.0,65.1,65.9,-0.4598,0.073,0.0,1.0,20.47,21.02,1.34,1.34,null,0.49,3.07,3.53,-0.3,0.0,65.1,71.6,58.6,null,null,null,"""OTC"""
"""1264""",2015-06-01,226.0,226.0,219.5,222.0,74.0,16443.0,-1.7699,0.2416,30633.0,6800.0,222.0,224.0,-1.7858,0.237,0.067,62.0,18.06,19.52,4.04,4.42,null,1.88,3.6,4.5,-4.0,2.8761,222.0,244.0,200.0,null,null,null,"""OTC"""
"""1336""",2015-06-01,10.7,10.7,10.5,10.55,36.0,378.0,0.4762,0.0574,62753.0,662.0,10.5,10.55,0.4751,0.023,0.002,13.0,null,null,0.44,0.44,null,0.34,0.76,0.0,0.05,1.9048,10.55,11.6,9.5,null,null,null,"""OTC"""
"""1565""",2015-06-01,547.0,550.0,538.0,541.0,104.0,56789.0,-0.3683,0.2063,50417.0,27275.0,541.0,543.0,-0.369,0.949,0.233,103.0,20.06,20.05,5.89,5.89,null,5.08,4.16,4.41,-2.0,2.2099,541.0,595.0,487.0,null,null,null,"""OTC"""
"""1569""",2015-06-01,16.4,16.6,16.15,16.2,97.0,1584.0,-1.2195,0.1034,93785.0,1519.0,16.2,16.35,-1.227,0.053,0.007,74.0,null,null,0.68,0.76,null,0.57,4.94,0.0,-0.2,2.7439,16.2,17.8,14.6,null,null,null,"""OTC"""


In [11]:
# 找出 OHLCV_df 內有哪些 date 不在 calendar_df 裡
# 先確保 calendar_df 和 OHLCV_df 的 date 欄位型態一致
# 若 calendar_df 只有一欄 'date' 且型別為 date/datetime

ohlcv_dates = OHLCV_df['date'].unique()
calendar_dates = calendar_df['date'].unique()

missing_dates = ohlcv_dates.filter(~ohlcv_dates.is_in(calendar_dates.to_list()))

if len(missing_dates) > 0 :
    print(missing_dates)
else:
    print("OHLCV_df 內所有日期都在 calendar_df 裡")

OHLCV_df 內所有日期都在 calendar_df 裡


In [12]:
# 全額交割(Y) 或非 TSE/OTC → untradable=1；其餘明確為 0（避免 Kleene null）
OHLCV_df = OHLCV_df.with_columns(
    (
        (pl.col("full_delivery") == "Y").fill_null(False)
        | (~pl.col("market").is_in(["TSE", "OTC"])).fill_null(False)
    )
    .cast(pl.Int8)
    .alias("untradable")
)

### 處理暫停交易的日期
將暫停交易表的時間區段填入 OHLCV的欄位並將其標記為不可交易

In [13]:
# 停牌區間填入 OHLCV：[start_halt, resume)；另標 start 前一交易日 untradable
# avoid_early 只寫入「start_halt 前一交易日」（可提前避開的停牌）
halts = (
    trading_halt_df
    .select(["symbol", "start_halt", "resume_trading_date", "reason", "avoid_early"])
    .drop_nulls("start_halt")
    .join(OHLCV_df.select("symbol").unique(), on="symbol", how="inner")
)

halted = (
    OHLCV_df.select(["symbol", "date"])
    .join(halts, on="symbol", how="inner")
    .filter(
        (pl.col("date") >= pl.col("start_halt"))
        & (
            pl.col("resume_trading_date").is_null()
            | (pl.col("date") < pl.col("resume_trading_date"))
        )
    )
    .select(["symbol", "date"])
    .unique()
)

# start 前一交易日 + 該停牌事件的 avoid_early
pre_halt = (
    halts.select(["symbol", "start_halt", "avoid_early"])
    .join(calendar_df.select("date"), how="cross")
    .filter(pl.col("date") < pl.col("start_halt"))
    .group_by(["symbol", "start_halt"])
    .agg(
        pl.col("date").max().alias("date"),
        pl.col("avoid_early").first().alias("avoid_early"),
    )
    .drop_nulls("date")
)
# 同一 (symbol, date) 若對應多段停牌，取 max(avoid_early)
pre_halt_flag = (
    pre_halt
    .group_by(["symbol", "date"])
    .agg(pl.col("avoid_early").max().cast(pl.Int8).alias("avoid_early"))
)
pre_halt_days = pre_halt_flag.select(["symbol", "date"])

halted = (
    pl.concat([halted, pre_halt_days])
    .unique()
    .with_columns(pl.lit(1).cast(pl.Int8).alias("halt_flag"))
)

OHLCV_df = (
    OHLCV_df
    .join(halted, on=["symbol", "date"], how="left")
    .join(pre_halt_flag, on=["symbol", "date"], how="left")
    .with_columns(
        pl.max_horizontal(pl.col("untradable"), pl.col("halt_flag").fill_null(0))
        .cast(pl.Int8)
        .alias("untradable"),
        pl.col("avoid_early").fill_null(0).cast(pl.Int8).alias("avoid_early"),
    )
    .drop("halt_flag")
)

print("halted rows:", halted.height, "(含 start 前一交易日)")
print(
    "avoid_early=1 列數:",
    OHLCV_df.filter(pl.col("avoid_early") == 1).height,
    "（應落在 start 前一交易日）",
)
OHLCV_df.filter(pl.col("avoid_early") == 1).head()


halted rows: 6663 (含 start 前一交易日)
avoid_early=1 列數: 558 （應落在 start 前一交易日）


symbol,date,open,high,low,close,volume,amount,return_day,turnover,shares_outstanding_thousand,market_value_million,last_bid,last_ask,log_return_day,market_value_pct,trading_value_pct,numtrades,pe_ratio_tse,pe_ratio_tej,pb_ratio_tse,pb_ratio_tej,limit_status,ps_ratio_tej,div_yield_tse,cash_div_yield,price_change,spread,next_open_ref,next_limit_up,next_limit_down,attention_stock,disposed_stock,full_delivery,market,untradable,avoid_early
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,i8,i8
"""3105""",2015-08-17,28.35,29.1,27.5,27.5,8739.0,246624.0,-2.8269,1.1744,744136.0,20464.0,27.5,27.55,-2.8676,0.895,1.783,3957.0,8.62,8.65,1.23,1.23,null,1.85,0.73,0.73,-0.8,5.6537,27.5,30.25,24.75,null,null,null,"""OTC""",1,1
"""4107""",2015-08-19,85.0,85.7,80.5,82.1,993.0,81880.0,-3.4118,1.1552,85957.0,7057.0,82.1,82.3,-3.4713,0.32,0.463,750.0,23.39,23.44,3.61,3.62,null,6.03,2.16,3.18,-2.9,6.1176,82.1,90.3,73.9,null,null,null,"""OTC""",1,1
"""4951""",2015-09-02,10.23,10.23,10.23,10.23,0.0,0.0,0.0,0.0,39212.0,401.0,10.21,10.73,0.0,null,null,0.0,null,12.69,null,0.66,null,1.38,null,4.91,0.0,0.0,0.0,null,null,null,null,null,"""REG""",1,1
"""3555""",2015-09-04,3.36,3.36,3.07,3.07,49.0,151.0,-5.5385,0.0895,54778.0,168.0,3.07,3.08,-5.6977,0.008,0.001,35.0,null,null,1.27,1.08,null,0.51,0.0,0.0,-0.18,8.9231,3.07,3.37,2.77,null,null,"""Y""","""OTC""",1,1
"""1742""",2015-09-11,11.8,11.8,11.8,11.8,191.0,2253.0,-9.9237,0.2953,64691.0,763.0,null,11.8,-10.4513,0.033,0.01,40.0,null,null,1.07,1.04,"""-""",0.76,25.42,0.0,-1.3,0.0,11.8,12.95,10.65,null,null,"""Y""","""OTC""",1,1


In [14]:
# 處理過的暫停交易資料的原因統計
# halted 無 reason，以 halts 為主；只保留 start_halt 落在 halted 且 avoid_early==0 的事件
reason_stats = (
    halts
    .filter(pl.col("avoid_early") == 0)
    .join(
        halted.select(["symbol", pl.col("date").alias("start_halt")]).unique(),
        on=["symbol", "start_halt"],
        how="semi",
    )
    .group_by("reason")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
# 不統計內容包含 "因有重大訊息待公佈" 的數據
# 這些內容都是無法提前知道的
exclude_keywords = ["重大訊息待公佈", "重大訊息待公布", "重大事項待公布", "重大訊息", "重大事項", "重大消息"]
reason_stats = reason_stats.filter(~pl.col("reason").str.contains('|'.join(exclude_keywords)))

print("各暫停交易理由的統計：")
# 計算 reason_stats 的 count 欄位和
total_count = reason_stats["count"].sum()
print("count 欄位總和:", total_count)

# 列出剩餘理由的原始資料
filtered_reasons = reason_stats["reason"].to_list()
filtered_halts = halts.filter(
    (pl.col("avoid_early") == 0) &
    (pl.col("reason").is_in(filtered_reasons))
).select(["symbol", "start_halt", "resume_trading_date", "reason"])
print("全部過濾後 halt 資料：")
#print(filtered_halts)

# 【新增】指定條件的篩選：指定標的(specific_symbols)與指定 start_halt 日期(specific_start_halts)
# 請自行指定你要查的範例

# 缺少指定的key word 無法判定 手工處理
# 4746 20220809 公布訊息
# 6838 20220809 公布訊息
# 7772 20260420 漲84% 熔斷
# 5225 20210315 公布訊息
# 6224 20200708 公布訊息
# 6885 20231222 跌60% 
# 6589 20210323 公布訊息
# 2102 20230210 公布訊息
# 4414 20180226 公布訊息
# 4743 20200415 公布訊息
# 2474 20200818 公布訊息
# 6438 20210415 公布訊息
# 5432 20191001 公布訊息
specific_symbols = [
    "4746", "7772" , "5225" , "6838" , "6224" , "6885" , "6589" , "2102" , "4414" , "4743" , "2474" , "6438" , "5432"
    ]  # 這裡可替換為你要查的標的代碼
specific_start_halts = [
    "2022-08-09" , "2026-04-20" , "2021-03-15" , "2022-08-09" , "2020-07-08" , "2023-12-22" , "2021-03-23" , "2023-02-10" , 
    "2018-02-26" , "2020-04-15" , "2020-08-18" , "2021-04-15" , "2019-10-01"
    ]  # 替換為你要查的 start_halt 日期(字串格式、可多個)
filtered_specific = filtered_halts.filter(
    ~(
        pl.col("symbol").is_in(specific_symbols) &
        pl.col("start_halt").cast(pl.Utf8).is_in(specific_start_halts)
    )
)

print("指定標的、指定start_halt的篩選結果：")

filtered_specific

各暫停交易理由的統計：
count 欄位總和: 13
全部過濾後 halt 資料：
指定標的、指定start_halt的篩選結果：


symbol,start_halt,resume_trading_date,reason
str,date,date,str


In [15]:
# 驗證：
# 1) [start_halt, resume) 皆 untradable==1
# 2) start 前一交易日亦 untradable==1
# 3) 恢復日可不在「自己的」暫停區間；若同日又是下一段 start（或下一段的 start 前一日），仍可暫停
expected_halted = (
    OHLCV_df.select(["symbol", "date", "untradable"])
    .join(halts, on="symbol", how="inner")
    .filter(
        (pl.col("date") >= pl.col("start_halt"))
        & (
            pl.col("resume_trading_date").is_null()
            | (pl.col("date") < pl.col("resume_trading_date"))
        )
    )
)
missing = expected_halted.filter(pl.col("untradable") != 1)

expected_pre_halt = (
    halts.select(["symbol", "start_halt"])
    .join(calendar_df.select("date"), how="cross")
    .filter(pl.col("date") < pl.col("start_halt"))
    .group_by(["symbol", "start_halt"])
    .agg(pl.col("date").max().alias("date"))
    .select(["symbol", "date"])
    .drop_nulls("date")
    .unique()
    .join(OHLCV_df.select(["symbol", "date", "untradable"]), on=["symbol", "date"], how="inner")
)
missing_pre = expected_pre_halt.filter(pl.col("untradable") != 1)

# 恢復日落在 expected_halted：多半是「恢復日 = 下一段 start」（銜接停牌），屬正確
resume_days = (
    halts.filter(pl.col("resume_trading_date").is_not_null())
    .select(["symbol", pl.col("resume_trading_date").alias("date")])
    .unique()
)
start_days = halts.select(["symbol", pl.col("start_halt").alias("date")]).unique()
pre_halt_days = expected_pre_halt.select(["symbol", "date"]).unique()
# 恢復日若等於下一段 start，或等於某段的 start 前一日，都算合理銜接
allowed_resume_halt = pl.concat([start_days, pre_halt_days]).unique()

resume_in_halted = (
    OHLCV_df.select(["symbol", "date"])
    .join(resume_days, on=["symbol", "date"], how="inner")
    .join(expected_halted.select(["symbol", "date"]).unique(), on=["symbol", "date"], how="inner")
)
resume_wrongly_in_halt = resume_in_halted.join(allowed_resume_halt, on=["symbol", "date"], how="anti")
chained_ok = resume_in_halted.join(allowed_resume_halt, on=["symbol", "date"], how="inner")

print("應暫停但未標記:", missing.height)
print("start前一日應暫停但未標記:", missing_pre.height)
print("恢復日銜接下一段/其前一日(正確仍暫停):", chained_ok.height)
print("恢復日被誤算進暫停區間:", resume_wrongly_in_halt.height)
print("暫停區間覆蓋 OHLCV 筆數:", expected_halted.select(["symbol", "date"]).unique().height)
print("start前一日覆蓋 OHLCV 筆數:", expected_pre_halt.height)

if missing.height == 0 and missing_pre.height == 0 and resume_wrongly_in_halt.height == 0:
    print("ok: 暫停區間與 start 前一日條件滿足")
else:
    if missing.height:
        print("missing:")
        display(missing.head())
    if missing_pre.height:
        print("missing_pre:")
        display(missing_pre.head())
    if resume_wrongly_in_halt.height:
        print("resume_wrongly_in_halt:")
        display(resume_wrongly_in_halt.head())
if chained_ok.height:
    print("銜接樣本:")
    display(chained_ok)

應暫停但未標記: 0
start前一日應暫停但未標記: 0
恢復日銜接下一段/其前一日(正確仍暫停): 4
恢復日被誤算進暫停區間: 0
暫停區間覆蓋 OHLCV 筆數: 5804
start前一日覆蓋 OHLCV 筆數: 859
ok: 暫停區間與 start 前一日條件滿足
銜接樣本:


symbol,date
str,date
"""8101""",2024-09-05
"""8101""",2024-09-16
"""6885""",2023-12-22
"""2025""",2016-02-15


## 處理復權因子
先標記價格跳空日期 並先處理現金除息

### 引入現金除息資料

In [16]:
import Cash_Dividends

cash_dividend_path = os.path.join(rebuild_root, "Cash_Dividends")
cash_dividend_df = read_tej_txt(
    one_txt(cash_dividend_path),
    Cash_Dividends.CASH_DIVIDENDS_SCHEMA,
).select(Cash_Dividends.keep_col)
cash_dividend_df.head()


symbol,date,dividend_per_share,prev_close,ex_div_ref_price
str,date,f64,f64,f64
"""5850""",2015-06-01,0.27768,null,null
"""8454""",2015-06-01,8.48,290.5,282.02
"""910322""",2015-06-01,0.5649525,32.85,32.28
"""910861""",2015-06-02,0.3929,27.35,26.95
"""000960""",2015-06-03,0.486281,null,null


#### 計算除息的復權因子

In [17]:
# date=除息日；adj_factor = prev_close / ex_div_ref_price（分母須 > 0）
# 分母 ≤0 或缺價 → adj_factor=null，並打印出來
_ok = (
    pl.col("prev_close").is_not_null()
    & (pl.col("prev_close") > 0)
    & pl.col("ex_div_ref_price").is_not_null()
    & (pl.col("ex_div_ref_price") > 0)
)
dividend_info = cash_dividend_df.with_columns(
    pl.when(_ok)
    .then(pl.col("prev_close") / pl.col("ex_div_ref_price"))
    .otherwise(None)
    .alias("adj_factor")
)
_bad = dividend_info.filter(pl.col("adj_factor").is_null())
if _bad.height:
    print(f"cash dividend adj_factor null：{_bad.height} 筆")
    print(_bad.select(["symbol", "date", "dividend_per_share", "prev_close", "ex_div_ref_price"]))
else:
    print("cash dividend adj_factor：無 null")
dividend_info.head()


cash dividend adj_factor null：623 筆
shape: (623, 5)
┌────────┬────────────┬────────────────────┬────────────┬──────────────────┐
│ symbol ┆ date       ┆ dividend_per_share ┆ prev_close ┆ ex_div_ref_price │
│ ---    ┆ ---        ┆ ---                ┆ ---        ┆ ---              │
│ str    ┆ date       ┆ f64                ┆ f64        ┆ f64              │
╞════════╪════════════╪════════════════════╪════════════╪══════════════════╡
│ 5850   ┆ 2015-06-01 ┆ 0.27768            ┆ null       ┆ null             │
│ 000960 ┆ 2015-06-03 ┆ 0.486281           ┆ null       ┆ null             │
│ 5839   ┆ 2015-06-10 ┆ 1.031669           ┆ null       ┆ null             │
│ 2837   ┆ 2015-06-11 ┆ 1.200804           ┆ null       ┆ null             │
│ 8476   ┆ 2015-06-11 ┆ 0.869923           ┆ null       ┆ null             │
│ …      ┆ …          ┆ …                  ┆ …          ┆ …                │
│ 5844   ┆ 2026-05-15 ┆ 1.244212           ┆ null       ┆ null             │
│ 2809   ┆ 2026-05-19 ┆ 

symbol,date,dividend_per_share,prev_close,ex_div_ref_price,adj_factor
str,date,f64,f64,f64,f64
"""5850""",2015-06-01,0.27768,null,null,null
"""8454""",2015-06-01,8.48,290.5,282.02,1.030069
"""910322""",2015-06-01,0.5649525,32.85,32.28,1.017658
"""910861""",2015-06-02,0.3929,27.35,26.95,1.014842
"""000960""",2015-06-03,0.486281,null,null,null


In [18]:
# 從OHLCV中的symbol欄位抓出所有unique的，以此為標準，篩除dividend_info中的symbol
valid_symbols = set(OHLCV_df["symbol"].unique())
dividend_info = dividend_info.filter(pl.col("symbol").is_in(valid_symbols))
dividend_info.head()

symbol,date,dividend_per_share,prev_close,ex_div_ref_price,adj_factor
str,date,f64,f64,f64,f64
"""8454""",2015-06-01,8.48,290.5,282.02,1.030069
"""8070""",2015-06-03,3.5,103.5,100.0,1.035
"""6026""",2015-06-04,0.5,11.94,11.44,1.043706
"""6219""",2015-06-04,0.718455,17.95,14.73,1.218601
"""1813""",2015-06-05,1.0,29.95,28.95,1.034542


#### NA 排查
原因是某些股票在不可交易的時間 依然有除息 所以會有紀錄 但是沒有開收盤價  
排查方法是將symbol,date形成一個pair去比較ohlcv 內有沒有這組

In [19]:
# 找出 dividend_info 中任意欄位有 NA 的 row，並確認是否存在於 OHLCV
na_rows = (
    dividend_info
    .filter(pl.any_horizontal(pl.col(c).is_null() for c in dividend_info.columns))
    .select(["symbol", "date"])
    .unique()
)
print("na_rows:", na_rows.height)
print(na_rows.head())

if na_rows.height == 0:
    has_pair_df = pl.DataFrame(
        schema={"symbol": pl.Utf8, "date": pl.Date, "exists_in_ohlcv": pl.Boolean}
    )
else:
    has_pair_df = (
        na_rows.join(
            OHLCV_df.select(["symbol", "date"]).unique().with_columns(pl.lit(True).alias("exists_in_ohlcv")),
            on=["symbol", "date"],
            how="left",
        )
        .with_columns(pl.col("exists_in_ohlcv").fill_null(False))
    )

print("check_na_pairs (只列出存在於 OHLCV 的)：")
print(has_pair_df.filter(pl.col("exists_in_ohlcv")))


na_rows: 126
shape: (5, 2)
┌────────┬────────────┐
│ symbol ┆ date       │
│ ---    ┆ ---        │
│ str    ┆ date       │
╞════════╪════════════╡
│ 6894   ┆ 2022-07-19 │
│ 7805   ┆ 2024-09-16 │
│ 6592   ┆ 2017-07-18 │
│ 6474   ┆ 2017-07-20 │
│ 2755   ┆ 2019-07-23 │
└────────┴────────────┘
check_na_pairs (只列出存在於 OHLCV 的)：
shape: (0, 3)
┌────────┬──────┬─────────────────┐
│ symbol ┆ date ┆ exists_in_ohlcv │
│ ---    ┆ ---  ┆ ---             │
│ str    ┆ date ┆ bool            │
╞════════╪══════╪═════════════════╡
└────────┴──────┴─────────────────┘


In [20]:
# 不存在於 OHLCV 的 (symbol, date) 直接 drop（停牌日仍有除息紀錄但無價）
no_pair = has_pair_df.filter(~pl.col("exists_in_ohlcv")).select(["symbol", "date"])
dividend_info = dividend_info.join(no_pair, on=["symbol", "date"], how="anti")

print("dividend_info after dropping rows not present in ohlcv:")
print(dividend_info.head())

still_na_rows = dividend_info.filter(
    pl.any_horizontal(pl.col(c).is_null() for c in dividend_info.columns)
)
if still_na_rows.height > 0:
    print("仍有任意欄位為 NA 的 rows：")
    print(still_na_rows.select(["symbol", "date"]).unique())
else:
    print("drop後，dividend_info 中已無任一欄位為 NA 的 row")


dividend_info after dropping rows not present in ohlcv:
shape: (5, 6)
┌────────┬────────────┬────────────────────┬────────────┬──────────────────┬────────────┐
│ symbol ┆ date       ┆ dividend_per_share ┆ prev_close ┆ ex_div_ref_price ┆ adj_factor │
│ ---    ┆ ---        ┆ ---                ┆ ---        ┆ ---              ┆ ---        │
│ str    ┆ date       ┆ f64                ┆ f64        ┆ f64              ┆ f64        │
╞════════╪════════════╪════════════════════╪════════════╪══════════════════╪════════════╡
│ 8454   ┆ 2015-06-01 ┆ 8.48               ┆ 290.5      ┆ 282.02           ┆ 1.030069   │
│ 8070   ┆ 2015-06-03 ┆ 3.5                ┆ 103.5      ┆ 100.0            ┆ 1.035      │
│ 6026   ┆ 2015-06-04 ┆ 0.5                ┆ 11.94      ┆ 11.44            ┆ 1.043706   │
│ 6219   ┆ 2015-06-04 ┆ 0.718455           ┆ 17.95      ┆ 14.73            ┆ 1.218601   │
│ 1813   ┆ 2015-06-05 ┆ 1.0                ┆ 29.95      ┆ 28.95            ┆ 1.034542   │
└────────┴────────────┴───────

In [21]:
# 在 OHLCV_df 新增 adj_factor，預設 1；除息日覆寫為 dividend 的 adj_factor
# 已知同日多筆配息（息值不同但 prev_close / ex_div_ref_price 相同）→ 指定後只留一筆
from datetime import date as _date

CASH_DIV_DUP_OK_PAIRS: list[tuple[str, _date]] = [
    ("2236", _date(2021, 9, 13)), # 同時發兩筆 ky公司 延遲
    ("2505", _date(2020, 8, 18)), # 季配息 + 土地出售
    ("2505", _date(2021, 11, 24)), # 兩季一起發
    ("2542", _date(2019, 10, 2)), # 2018 全年 + 2019改成季配
    ("2756", _date(2021, 8, 23)),  # 年配 + 季配
    ("4912", _date(2021, 7, 8)), # 20年底季配 + 21年初季配
]


OHLCV_df = OHLCV_df.with_columns(pl.lit(1.0).alias("adj_factor"))

_dup = (
    dividend_info.filter(pl.col("adj_factor").is_not_null())
    .group_by(["symbol", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .sort(["symbol", "date"])
)
if _dup.height:
    _ok_pair_df = pl.DataFrame(
        CASH_DIV_DUP_OK_PAIRS,
        schema={"symbol": pl.Utf8, "date": pl.Date},
        orient="row",
    ) if CASH_DIV_DUP_OK_PAIRS else pl.DataFrame(
        schema={"symbol": pl.Utf8, "date": pl.Date}
    )
    _allowed = _dup.join(_ok_pair_df, on=["symbol", "date"], how="semi")
    _not_allowed = _dup.join(_ok_pair_df, on=["symbol", "date"], how="anti")

    if _not_allowed.height:
        _show = dividend_info.join(
            _not_allowed.select(["symbol", "date"]), on=["symbol", "date"], how="inner"
        ).sort(["symbol", "date"])
        print("未在 CASH_DIV_DUP_OK_PAIRS 的重複（需人工處理）：")
        display(_not_allowed)
        display(_show)
        raise ValueError(
            f"cash dividend 有 {_not_allowed.height} 組未允許的重複 (symbol, date)"
        )

    _dup_rows = dividend_info.join(
        _allowed.select(["symbol", "date"]), on=["symbol", "date"], how="inner"
    )
    _price_bad = (
        _dup_rows.group_by(["symbol", "date"])
        .agg(
            pl.col("prev_close").n_unique().alias("n_prev"),
            pl.col("ex_div_ref_price").n_unique().alias("n_ref"),
            pl.col("adj_factor").n_unique().alias("n_adj"),
        )
        .filter(
            (pl.col("n_prev") > 1)
            | (pl.col("n_ref") > 1)
            | (pl.col("n_adj") > 1)
        )
    )
    if _price_bad.height:
        print("允許的重複組但 prev_close / ex_div_ref_price / adj_factor 不一致：")
        display(_price_bad)
        display(
            _dup_rows.join(_price_bad.select(["symbol", "date"]), on=["symbol", "date"], how="inner")
            .sort(["symbol", "date"])
        )
        raise ValueError("同日重複配息的價格欄不一致，無法自動去重")

    print(
        f"允許的同日重複配息 {_allowed.height} 組："
        f"prev_close/ex_div_ref_price 相同 → keep first"
    )
    display(_allowed)
    dividend_info = dividend_info.unique(subset=["symbol", "date"], keep="first")

merge_df = dividend_info.filter(pl.col("adj_factor").is_not_null()).select(
    ["symbol", "date", "adj_factor"]
)
_dup2 = merge_df.group_by(["symbol", "date"]).len().filter(pl.col("len") > 1)
if _dup2.height:
    raise ValueError(f"去重後仍有重複：{_dup2.height} 組")

OHLCV_df = (
    OHLCV_df.join(merge_df, on=["symbol", "date"], how="left")
    .with_columns(
        pl.coalesce(pl.col("adj_factor_right"), pl.col("adj_factor")).alias("adj_factor")
    )
    .drop("adj_factor_right")
)

print("OHLCV_df after merging adj_factor:")
OHLCV_df.head()





允許的同日重複配息 6 組：prev_close/ex_div_ref_price 相同 → keep first


symbol,date,len
str,date,u32
"""2236""",2021-09-13,2
"""2505""",2020-08-18,2
"""2505""",2021-11-24,2
"""2542""",2019-10-02,2
"""2756""",2021-08-23,2
"""4912""",2021-07-08,2


OHLCV_df after merging adj_factor:


symbol,date,open,high,low,close,volume,amount,return_day,turnover,shares_outstanding_thousand,market_value_million,last_bid,last_ask,log_return_day,market_value_pct,trading_value_pct,numtrades,pe_ratio_tse,pe_ratio_tej,pb_ratio_tse,pb_ratio_tej,limit_status,ps_ratio_tej,div_yield_tse,cash_div_yield,price_change,spread,next_open_ref,next_limit_up,next_limit_down,attention_stock,disposed_stock,full_delivery,market,untradable,avoid_early,adj_factor
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,i8,i8,f64
"""1259""",2015-06-01,65.1,65.1,65.1,65.1,1.0,65.0,-0.4587,0.0031,32390.0,2109.0,65.1,65.9,-0.4598,0.073,0.0,1.0,20.47,21.02,1.34,1.34,null,0.49,3.07,3.53,-0.3,0.0,65.1,71.6,58.6,null,null,null,"""OTC""",0,0,1.0
"""1264""",2015-06-01,226.0,226.0,219.5,222.0,74.0,16443.0,-1.7699,0.2416,30633.0,6800.0,222.0,224.0,-1.7858,0.237,0.067,62.0,18.06,19.52,4.04,4.42,null,1.88,3.6,4.5,-4.0,2.8761,222.0,244.0,200.0,null,null,null,"""OTC""",0,0,1.0
"""1336""",2015-06-01,10.7,10.7,10.5,10.55,36.0,378.0,0.4762,0.0574,62753.0,662.0,10.5,10.55,0.4751,0.023,0.002,13.0,null,null,0.44,0.44,null,0.34,0.76,0.0,0.05,1.9048,10.55,11.6,9.5,null,null,null,"""OTC""",0,0,1.0
"""1565""",2015-06-01,547.0,550.0,538.0,541.0,104.0,56789.0,-0.3683,0.2063,50417.0,27275.0,541.0,543.0,-0.369,0.949,0.233,103.0,20.06,20.05,5.89,5.89,null,5.08,4.16,4.41,-2.0,2.2099,541.0,595.0,487.0,null,null,null,"""OTC""",0,0,1.0
"""1569""",2015-06-01,16.4,16.6,16.15,16.2,97.0,1584.0,-1.2195,0.1034,93785.0,1519.0,16.2,16.35,-1.227,0.053,0.007,74.0,null,null,0.68,0.76,null,0.57,4.94,0.0,-0.2,2.7439,16.2,17.8,14.6,null,null,null,"""OTC""",0,0,1.0


### 引入 權 改變的數據
可能會有 除權 / 減資 等等的狀況

In [22]:
# 建立除權的 file 路徑
from Corporate_Actions import CORPORATE_ACTIONS_SCHEMA
ex_rights_file_path = os.path.join(rebuild_root, "Corporate_Actions")
ex_rights_df = read_tej_txt(
    one_txt(ex_rights_file_path),
    CORPORATE_ACTIONS_SCHEMA,
)
ex_rights_df.head()


symbol,date,cash_capital_increase_thousand,retained_earnings_increase_thousand,capital_reserve_increase_thousand,capital_reduction_thousand,capital_reduction_cash_per_share,capital_reduction_rate,par_value_change_shares_thousand,par_value_change_ratio,cash_subscription_rate,cash_subscription_price,retained_earnings_stock_rate,capital_reserve_stock_rate,ex_rights_ref_price,prev_close,stock_dividend_yn,private_placement_yn,total_capital_thousand,par_value,currency,total_shares_thousand,distributable_shares_thousand,employee_bonus_shares_thousand,cb_conversion_shares_thousand,pref_conversion_shares_thousand,share_swap_thousand,treasury_cancel_thousand,rsa_cancel_thousand,merger_shares_thousand,transfer_shares_thousand,eso_shares_thousand,rsa_shares_thousand,other_shares_thousand,employee_stock_rate,cb_conversion_price,employee_subscription_rate,public_underwriting_rate,book_building_rate,gdr_rate,specific_rate,capital_change_announce_date,shareholder_meeting_date,sfc_approval_date,book_closure_start,book_closure_end,shareholder_payment_start,shareholder_payment_end,cash_call_start,cash_call_end,payment_certificate_announce_date,payment_certificate_issue_date,payment_certificate_list_date,cash_increase_stock_announce_date,cash_increase_stock_issue_date,cash_increase_stock_list_date,rights_certificate_announce_date,rights_certificate_issue_date,rights_certificate_list_date,employee_bonus_announce_date,employee_bonus_issue_date,employee_bonus_list_date,stock_dividend_announce_date,stock_dividend_issue_date,stock_dividend_list_date,last_transfer_date,margin_buy_suspend_start,margin_suspend_end,margin_sell_suspend_start,short_cover_deadline,cash_increase_failed,capital_increase_fail_reason,other_note,capital_reduction_refund_date
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,str,str,str,date
"""1268""",2015-05-31,800.0,0.0,0.0,0.0,0.0,null,0.0,null,null,60.0,null,null,null,null,"""N""","""N""",290000.0,10.0,"""NTD""",29000.0,28200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null
"""2745""",2015-05-31,1000.0,0.0,0.0,0.0,0.0,null,0.0,null,null,40.0,null,null,null,null,"""N""","""N""",243442.0,10.0,"""NTD""",24344.0,23344.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null
"""4148""",2015-05-31,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,null,null,null,"""N""","""N""",565000.0,10.0,"""NTD""",56500.0,55000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1500.0,0.0,0.0,null,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,"""員""",null
"""4767""",2015-05-31,1348.0,0.0,0.0,0.0,0.0,null,0.0,null,null,10.0,null,null,null,null,"""N""","""N""",255000.0,10.0,"""NTD""",25500.0,24152.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null
"""6244""",2015-05-31,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,null,null,null,"""N""","""N""",4.412334e6,10.0,"""NTD""",441233.0,441256.0,0.0,0.0,0.0,0.0,0.0,23.0,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,null,null,null,2015-05-11,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null

In [23]:
# 只保留 OHLCV 有 (symbol, date) 的除權列（無價日無法掛 adj_factor）
# 私募不做公開除權參考價調整 → 排除；配股（盈餘/公積增資）要留，交給下一格用數量欄篩
ex_rights_df = ex_rights_df.join(
    OHLCV_df.select(["symbol", "date"]).unique(),
    on=["symbol", "date"],
    how="semi",
)
ex_rights_df = ex_rights_df.filter(
    pl.col("private_placement_yn").fill_null("N") != "Y"
)


#### 配股（盈餘／公積增資）


In [24]:
# 除權復權：只取盈餘增資 / 公積增資（配股，證交所有除權參考價）
# 現金增資（現增認購）不走 prev_close/ex_rights_ref；私募已在上一格排除
# 保留 ref_price 供後續與現金除息同日比對
has_stock_dividend_increase = (
    (
        pl.col("retained_earnings_increase_thousand").is_not_null()
        & (pl.col("retained_earnings_increase_thousand") != 0)
    )
    | (
        pl.col("capital_reserve_increase_thousand").is_not_null()
        & (pl.col("capital_reserve_increase_thousand") != 0)
    )
)
has_prices = (
    pl.col("ex_rights_ref_price").is_not_null()
    & (pl.col("ex_rights_ref_price") > 0)
    & pl.col("prev_close").is_not_null()
    & (pl.col("prev_close") > 0)
)

ex_rights_adj_factor_df = (
    ex_rights_df
    .filter(has_stock_dividend_increase & has_prices)
    .select([
        "symbol",
        "date",
        (pl.col("prev_close") / pl.col("ex_rights_ref_price")).alias("adj_factor"),
        pl.col("ex_rights_ref_price").alias("ref_price"),
    ])
    .sort(["symbol", "date"])
)

_dup = (
    ex_rights_adj_factor_df.group_by(["symbol", "date"]).len().filter(pl.col("len") > 1)
)
if _dup.height:
    display(_dup)
    display(ex_rights_adj_factor_df.join(_dup.select(["symbol", "date"]), on=["symbol", "date"], how="inner"))
    raise ValueError(f"ex_rights_adj_factor_df 有 {_dup.height} 組重複 (symbol, date)")
print("ex_rights_adj_factor_df（配股）唯一 OK，筆數:", ex_rights_adj_factor_df.height)
ex_rights_adj_factor_df.head()


ex_rights_adj_factor_df（配股）唯一 OK，筆數: 2008


symbol,date,adj_factor,ref_price
str,date,f64,f64
"""1101""",2018-07-26,1.138027,39.63
"""1101""",2019-08-13,1.157188,37.98
"""1101""",2020-08-12,1.109785,41.9
"""1101""",2022-07-21,1.127066,36.91
"""1103""",2023-07-27,1.034901,20.63


In [25]:
# 將 ex_rights_adj_factor_df 篩出的 (symbol, date) pair 從 ex_rights_df 過濾掉
exclude_pairs = ex_rights_adj_factor_df.select(["symbol", "date"])
ex_rights_df = ex_rights_df.join(exclude_pairs, on=["symbol", "date"], how="anti")
ex_rights_df

symbol,date,cash_capital_increase_thousand,retained_earnings_increase_thousand,capital_reserve_increase_thousand,capital_reduction_thousand,capital_reduction_cash_per_share,capital_reduction_rate,par_value_change_shares_thousand,par_value_change_ratio,cash_subscription_rate,cash_subscription_price,retained_earnings_stock_rate,capital_reserve_stock_rate,ex_rights_ref_price,prev_close,stock_dividend_yn,private_placement_yn,total_capital_thousand,par_value,currency,total_shares_thousand,distributable_shares_thousand,employee_bonus_shares_thousand,cb_conversion_shares_thousand,pref_conversion_shares_thousand,share_swap_thousand,treasury_cancel_thousand,rsa_cancel_thousand,merger_shares_thousand,transfer_shares_thousand,eso_shares_thousand,rsa_shares_thousand,other_shares_thousand,employee_stock_rate,cb_conversion_price,employee_subscription_rate,public_underwriting_rate,book_building_rate,gdr_rate,specific_rate,capital_change_announce_date,shareholder_meeting_date,sfc_approval_date,book_closure_start,book_closure_end,shareholder_payment_start,shareholder_payment_end,cash_call_start,cash_call_end,payment_certificate_announce_date,payment_certificate_issue_date,payment_certificate_list_date,cash_increase_stock_announce_date,cash_increase_stock_issue_date,cash_increase_stock_list_date,rights_certificate_announce_date,rights_certificate_issue_date,rights_certificate_list_date,employee_bonus_announce_date,employee_bonus_issue_date,employee_bonus_list_date,stock_dividend_announce_date,stock_dividend_issue_date,stock_dividend_list_date,last_transfer_date,margin_buy_suspend_start,margin_suspend_end,margin_sell_suspend_start,short_cover_deadline,cash_increase_failed,capital_increase_fail_reason,other_note,capital_reduction_refund_date
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,str,str,str,date
"""6244""",2015-06-01,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,null,null,null,"""N""","""N""",4.869054e6,10.0,"""NTD""",486905.0,441233.0,0.0,0.0,0.0,0.0,0.0,0.0,45672.0,0.0,0.0,0.0,0.0,null,0.0,null,null,null,null,null,2015-05-11,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2015-05-11,2015-06-01,2015-06-01,null,null,null,2015-07-17,2015-07-23,2015-07-23,null,null,null,null,null,"""N""",null,"""1:6(合併聯景)""",null
"""5225""",2015-06-02,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,null,null,null,"""N""","""N""",610020.0,10.0,"""NTD""",61002.0,60750.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,252.0,0.0,0.0,null,0.0,null,null,null,null,null,2015-05-12,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2015-06-15,2015-06-30,2015-06-30,null,null,null,null,null,"""N""",null,"""限制員工權利新股""",null
"""8027""",2015-06-03,5100.0,0.0,0.0,0.0,0.0,null,0.0,null,null,30.0,null,null,null,null,"""N""","""N""",559090.0,10.0,"""NTD""",55909.0,50809.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,85.0,0.0,0.0,0.0,2015-05-27,2014-06-25,null,null,null,null,2015-06-03,null,null,2015-05-27,2015-06-09,2015-06-09,2015-07-16,2015-07-20,2015-07-20,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null
"""2637""",2015-06-05,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,null,null,null,"""N""","""N""",4.983244e6,10.0,"""NTD""",498324.0,486808.0,0.0,11516.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,33.7,null,null,null,null,null,2015-06-11,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,"""cb1(26371)""",null
"""2395""",2015-06-08,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,null,null,null,"""N""","""N""",6.316636e6,10.0,"""NTD""",631664.0,631519.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,145.0,0.0,0.0,null

#### 減資

In [26]:
# 減資列：觀察價格齊全比例；無資料時不除法
has_capital_reduction = ex_rights_df.filter(
    (pl.col("capital_reduction_thousand").is_not_null())
    & (pl.col("capital_reduction_thousand") > 0)
)

total_rows = has_capital_reduction.height
has_both_prices = has_capital_reduction.filter(
    pl.col("ex_rights_ref_price").is_not_null() & pl.col("prev_close").is_not_null()
)
count_has_both_prices = has_both_prices.height

print(f"capital_reduction_thousand 有值 (非空且大於0) 的資料筆數: {total_rows}")
print(f"其中 ex_rights_ref_price 和 prev_close 皆不為空的筆數: {count_has_both_prices}")
if total_rows == 0:
    print("無減資資料，跳過比例計算")
else:
    print(f"比例: {count_has_both_prices / total_rows:.2%}")


capital_reduction_thousand 有值 (非空且大於0) 的資料筆數: 500
其中 ex_rights_ref_price 和 prev_close 皆不為空的筆數: 500
比例: 100.00%


In [27]:
# 減資事件落在停牌區間 [start_halt, resume) 者：附上起迄日 + adj_factor
_ok = (
    pl.col("prev_close").is_not_null()
    & (pl.col("prev_close") > 0)
    & pl.col("ex_rights_ref_price").is_not_null()
    & (pl.col("ex_rights_ref_price") > 0)
)
_cap = has_capital_reduction.with_columns(
    pl.when(_ok)
    .then(pl.col("prev_close") / pl.col("ex_rights_ref_price"))
    .otherwise(None)
    .alias("adj_factor"),
    pl.col("ex_rights_ref_price").alias("ref_price"),
)

_bad = _cap.filter(pl.col("adj_factor").is_null())
if _bad.height:
    print(f"capital reduction adj_factor null：{_bad.height} 筆（仍可進停牌對照，但無因子）")
    display(_bad.select(["symbol", "date", "prev_close", "ex_rights_ref_price"]))

cap_red_pairs = _cap.select(
    ["symbol", "date", "adj_factor", "ref_price"]
).with_row_index("row_idx")

joined = cap_red_pairs.join(
    trading_halt_df.select(["symbol", "start_halt", "resume_trading_date"]),
    on="symbol",
    how="inner",
)
in_suspension = joined.filter(
    (pl.col("date") >= pl.col("start_halt"))
    & (
        pl.col("resume_trading_date").is_null()
        | (pl.col("date") < pl.col("resume_trading_date"))
    )
)

# 落在停牌區間的減資：事件日 + 停牌起迄 + 調整因子
capital_reduction_halt_df = (
    in_suspension
    .select([
        "symbol",
        pl.col("date").alias("event_date"),  # 減資／除權基準日（落在停牌內）
        "start_halt",
        "resume_trading_date",
        "adj_factor",
        "ref_price",
    ])
    .unique()
    .sort(["symbol", "event_date", "start_halt"])
)

print(
    f"減資落在停牌區間：{capital_reduction_halt_df.height} 筆；"
    f"有恢復日：{capital_reduction_halt_df.filter(pl.col('resume_trading_date').is_not_null()).height}；"
    f"有 adj_factor：{capital_reduction_halt_df.filter(pl.col('adj_factor').is_not_null()).height}"
)
capital_reduction_halt_df


減資落在停牌區間：500 筆；有恢復日：500；有 adj_factor：500


symbol,event_date,start_halt,resume_trading_date,adj_factor,ref_price
str,date,date,date,f64,f64
"""1109""",2015-10-16,2015-10-16,2015-10-30,0.995238,10.5
"""1109""",2017-10-19,2017-10-19,2017-10-30,0.989788,12.73
"""1109""",2018-10-11,2018-10-11,2018-10-22,0.983333,15.0
"""1236""",2023-09-27,2023-09-27,2023-10-11,0.886022,23.25
"""1315""",2020-10-15,2020-10-15,2020-10-26,0.742254,71.0
…,…,…,…,…,…
"""9928""",2022-03-24,2022-03-24,2022-04-06,0.403082,16.87
"""9929""",2020-12-03,2020-12-03,2020-12-14,0.874486,14.58
"""9943""",2022-09-15,2022-09-15,2022-09-26,0.873926,62.82


In [28]:
# 供 OHLCV join 的減資因子表：
# - 落在停牌區間且有恢復日 → 掛在 resume_trading_date（停牌日 OHLCV 無列）
# - 其餘有 adj 的減資 → 掛在事件日 date
_suspended = (
    capital_reduction_halt_df
    .filter(
        pl.col("resume_trading_date").is_not_null()
        & pl.col("adj_factor").is_not_null()
    )
    .select([
        "symbol",
        pl.col("resume_trading_date").alias("date"),
        "adj_factor",
        "ref_price",
        pl.col("event_date"),
        "start_halt",
        "resume_trading_date",
    ])
)

_in_halt_events = capital_reduction_halt_df.select(
    ["symbol", pl.col("event_date").alias("date")]
).unique()

_not_suspended = (
    _cap.filter(pl.col("adj_factor").is_not_null())
    .join(_in_halt_events, on=["symbol", "date"], how="anti")
    .select([
        "symbol",
        "date",
        "adj_factor",
        "ref_price",
        pl.col("date").alias("event_date"),
        pl.lit(None).cast(pl.Date).alias("start_halt"),
        pl.lit(None).cast(pl.Date).alias("resume_trading_date"),
    ])
)

capital_reduction_only_df = (
    pl.concat([_suspended, _not_suspended], how="diagonal_relaxed")
    .select(["symbol", "date", "adj_factor", "ref_price", "event_date", "start_halt", "resume_trading_date"])
    .sort(["symbol", "date"])
)

print(
    f"capital_reduction_only_df：{_suspended.height} 筆掛恢復日、"
    f"{_not_suspended.height} 筆掛事件日、合計 {capital_reduction_only_df.height}"
)
capital_reduction_only_df.head()



capital_reduction_only_df：500 筆掛恢復日、0 筆掛事件日、合計 500


symbol,date,adj_factor,ref_price,event_date,start_halt,resume_trading_date
str,date,f64,f64,date,date,date
"""1109""",2015-10-30,0.995238,10.5,2015-10-16,2015-10-16,2015-10-30
"""1109""",2017-10-30,0.989788,12.73,2017-10-19,2017-10-19,2017-10-30
"""1109""",2018-10-22,0.983333,15.0,2018-10-11,2018-10-11,2018-10-22
"""1236""",2023-10-11,0.886022,23.25,2023-09-27,2023-09-27,2023-10-11
"""1315""",2020-10-26,0.742254,71.0,2020-10-15,2020-10-15,2020-10-26


In [29]:
# 減資 (symbol, date) 必須唯一，否則後續 join 會膨脹列數
_dup = (
    capital_reduction_only_df
    .group_by(["symbol", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .sort(["symbol", "date"])
)
if _dup.height:
    print("capital_reduction_only_df 重複 (symbol, date)：")
    display(_dup)
    display(
        capital_reduction_only_df.join(
            _dup.select(["symbol", "date"]), on=["symbol", "date"], how="inner"
        ).sort(["symbol", "date"])
    )
    raise ValueError(
        f"capital_reduction_only_df 有 {_dup.height} 組重複 (symbol, date)"
    )
print(
    "capital_reduction_only_df（減資）唯一 OK，筆數:",
    capital_reduction_only_df.height,
)
capital_reduction_only_df


capital_reduction_only_df（減資）唯一 OK，筆數: 500


symbol,date,adj_factor,ref_price,event_date,start_halt,resume_trading_date
str,date,f64,f64,date,date,date
"""1109""",2015-10-30,0.995238,10.5,2015-10-16,2015-10-16,2015-10-30
"""1109""",2017-10-30,0.989788,12.73,2017-10-19,2017-10-19,2017-10-30
"""1109""",2018-10-22,0.983333,15.0,2018-10-11,2018-10-11,2018-10-22
"""1236""",2023-10-11,0.886022,23.25,2023-09-27,2023-09-27,2023-10-11
"""1315""",2020-10-26,0.742254,71.0,2020-10-15,2020-10-15,2020-10-26
…,…,…,…,…,…,…
"""9928""",2022-04-06,0.403082,16.87,2022-03-24,2022-03-24,2022-04-06
"""9929""",2020-12-14,0.874486,14.58,2020-12-03,2020-12-03,2020-12-14
"""9943""",2022-09-26,0.873926,62.82,2022-09-15,2022-09-15,2022-09-26


## 籌碼數據

### 籌碼動向＆報酬(2008年起)

In [30]:
import institutional_return
institutional_return_path = os.path.join(rebuild_root, "institutional_return")
institutional_return_df = read_tej_txt(
    one_txt(institutional_return_path),
    institutional_return.INSTITUTIONAL_RETURN_SCHEMA,
).select(institutional_return.keep_col)
institutional_return_df.head()


symbol,date,foreign_net_buy_lot,it_net_buy_lot,inst_net_buy_lot,margin_net_buy_lot,cash_net_buy_lot,foreign_net_buy_days,it_net_buy_days,dealer_net_buy_days,inst_net_buy_days,margin_net_buy_days,cash_net_buy_days,foreign_streak_return,it_streak_return,dealer_streak_return,inst_streak_return,margin_streak_return,cash_streak_return,foreign_streak_cum_lot,it_streak_cum_lot,dealer_streak_cum_lot,inst_streak_cum_lot,margin_streak_cum_lot,cash_streak_cum_lot,foreign_streak_cum_amt_thousand,it_streak_cum_amt_thousand,dealer_streak_cum_amt_thousand,inst_streak_cum_amt_thousand,margin_streak_cum_amt_thousand,cash_streak_cum_amt_thousand
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""0050""",2015-06-01,-2658.0,-372.0,-4193.0,307.0,3886.0,-2.0,-1.0,-1.0,-1.0,2.0,1.0,-1.3986,-0.8439,-0.8439,-0.8439,-1.3986,-0.8439,-3112.0,-372.0,-1163.0,-4193.0,327.0,3886.0,-219719.0,-26220.0,-81974.0,-295543.0,23065.0,273904.0
"""0051""",2015-06-01,0.0,0.0,-23.0,-1.0,24.0,null,null,-5.0,-5.0,-1.0,5.0,0.0,0.0,1.0527,1.0527,0.0,1.0527,0.0,0.0,-64.0,-64.0,-1.0,63.0,0.0,0.0,-1920.0,-1920.0,-30.0,1890.0
"""0052""",2015-06-01,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,-0.9936,-0.9936,-0.9936,-0.9936,-0.9936,-0.9936,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""0053""",2015-06-01,0.0,0.0,-2.0,0.0,2.0,null,null,-1.0,-1.0,null,1.0,-1.3333,-1.3333,-1.3333,-1.3333,-1.3333,-1.3333,0.0,0.0,-2.0,-2.0,0.0,2.0,0.0,0.0,-59.0,-59.0,0.0,59.0
"""0054""",2015-06-01,0.0,0.0,40.0,0.0,-40.0,null,null,6.0,6.0,null,-6.0,-1.1213,-1.1213,-0.0421,-0.0421,-1.1213,-0.0421,0.0,0.0,81.0,81.0,0.0,-80.0,0.0,0.0,1927.0,1927.0,0.0,-1903.0


### 三大法人買賣超

In [31]:
import institutional_net_flow

institutional_net_flow_path = os.path.join(rebuild_root, "institutional_net_flow")
institutional_net_flow_df = read_tej_txt(
    one_txt(institutional_net_flow_path),
    institutional_net_flow.INSTITUTIONAL_NET_FLOW_SCHEMA,
).select(institutional_net_flow.keep_col)
institutional_net_flow_df.head()


symbol,date,foreign_holding_pct,it_holding_pct,dealer_holding_pct,foreign_turnover_pct,it_turnover_pct,dealer_turnover_pct,dealer_prop_net_buy_thousand_shares,dealer_hedge_net_buy_thousand_shares
str,date,f64,f64,f64,f64,f64,f64,f64,f64
"""0052""",2015-06-01,14.91,0.0,0.82,0.0,0.0,0.02,0.0,0.0
"""0050""",2015-06-01,11.42,0.25,19.61,0.13,0.03,0.3,46.0,-1209.0
"""0055""",2015-06-01,7.28,0.0,9.29,0.0,0.2,0.18,-75.0,288.0
"""0051""",2015-06-01,5.38,0.0,3.53,0.0,0.0,0.06,0.0,-23.0
"""0058""",2015-06-01,7.06,0.0,7.1,0.0,0.0,0.02,0.0,0.0


### 融資融券

In [32]:
import Credit_Transactions

Credit_Transaction_path = os.path.join(rebuild_root, "Credit_Transactions")
Credit_Transaction_df = Credit_Transactions.add_credit_ratios(
    read_tej_txt(
        one_txt(Credit_Transaction_path),
        Credit_Transactions.CREDIT_TRANSACTIONS_SCHEMA,
    )
).select(Credit_Transactions.keep_col)
Credit_Transaction_df.head()


symbol,date,marginable,margin_usage_rate,short_usage_rate,short_to_margin_ratio,margin_change_ratio,short_change_ratio,margin_volume_pct,short_volume_pct,margin_maintenance_ratio,short_maintenance_ratio,account_maintenance_ratio,margin_buy_lot,margin_sell_lot,margin_cash_repay_lot,short_buy_lot,short_sell_lot,short_stock_repay_lot,short_last_cover_date,margin_cash_repay_ratio,short_stock_repay_ratio
str,date,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,f64,f64
"""0050""",2015-06-01,"""Y""",1.82,0.62,34.1,6.91,-0.9,2.48,0.33,166.07,187.87,173.95,341.0,48.0,0.0,33.0,19.0,0.0,null,0.0,0.0
"""0051""",2015-06-01,"""Y""",0.04,0.0,0.0,-33.33,0.0,1.52,0.0,166.07,null,166.07,0.0,1.0,0.0,0.0,0.0,0.0,null,0.0,null
"""0052""",2015-06-01,"""Y""",0.0,0.0,null,0.0,0.0,0.0,0.0,null,null,null,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
"""0053""",2015-06-01,"""Y""",0.11,0.0,0.0,0.0,0.0,0.0,0.0,162.85,null,162.85,0.0,0.0,0.0,0.0,0.0,0.0,null,0.0,null
"""0054""",2015-06-01,"""Y""",0.02,0.0,0.0,0.0,0.0,0.0,0.0,164.11,null,164.11,0.0,0.0,0.0,0.0,0.0,0.0,null,0.0,null


### 當沖資料

In [33]:
import Day_Trading_Stats

Day_Trading_Stats_path = os.path.join(rebuild_root, "Day_Trading_Stats")
Day_Trading_Stats_df = read_tej_txt(
    one_txt(Day_Trading_Stats_path),
    Day_Trading_Stats.DAY_TRADING_STATS_SCHEMA,
).select(Day_Trading_Stats.keep_col)
Day_Trading_Stats_df.head()


symbol,date,day_trade_volume_mkt_pct,day_trade_margin_short_offset_pct
str,date,f64,f64
"""0050""",2015-06-01,2.99,0.15
"""0051""",2015-06-01,0.0,0.0
"""0053""",2015-06-01,11.11,0.0
"""0052""",2015-06-01,0.0,0.0
"""0056""",2015-06-01,0.32,0.0


### 借券餘額

In [34]:
import Securities_Lending

Securities_Lending_path = os.path.join(rebuild_root, "Securities_Lending")
Securities_Lending_df = Securities_Lending.add_lend_net_flow(
    read_tej_txt(
        one_txt(Securities_Lending_path),
        Securities_Lending.SECURITIES_LENDING_SCHEMA,
    )
).select(Securities_Lending.keep_col)
Securities_Lending_df.head()


symbol,date,lend_exch_balance,lend_broker_balance,lend_exch_net,lend_broker_net
str,date,f64,f64,f64,f64
"""0050""",2015-06-01,9000.0,2730.0,1000.0,0.0
"""006206""",2015-06-01,28.0,1302.0,0.0,0.0
"""0061""",2015-06-01,1925.0,11276.0,0.0,0.0
"""00638R""",2015-06-01,null,500.0,null,0.0
"""006205""",2015-06-01,14349.0,1713.0,0.0,0.0


### 籌碼分佈

In [35]:
import share_distribution

share_distribution_path = os.path.join(rebuild_root, "share_distribution")
share_distribution_df = read_tej_folder(
    share_distribution_path,
    share_distribution.SHARE_DISTRIBUTION_SCHEMA,
    allow_market_split=True,
).select(share_distribution.keep_col)
share_distribution_df.head()


symbol,date,foreign_trade_share_pct,it_trade_share_pct,dealer_trade_share_pct,credit_trade_share_pct,cash_trade_share_pct,free_float_turnover_pct,foreign_holding_ex_dir_pct,director_holding_pct,director_pledge_pct,margin_short_offset_pct,margin_increase_lot,margin_decrease_lot,short_increase_lot,short_decrease_lot
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""1101""",2015-06-01,57.34,0.0,1.4,0.04,41.22,0.19,30.44,12.04,20.24,0.02,168.0,0.0,28.0,0.0
"""1102""",2015-06-01,49.58,0.0,0.03,0.03,50.36,0.29,19.31,29.35,4.5,0.01,198.0,0.0,0.0,9.0
"""1103""",2015-06-01,13.7,0.0,0.0,2.37,83.93,0.03,5.73,18.75,0.0,2.28,29.0,0.0,1.0,0.0
"""1104""",2015-06-01,14.83,0.0,0.38,1.27,83.52,0.07,16.74,37.4,0.0,1.14,31.0,0.0,0.0,0.0
"""1108""",2015-06-01,16.12,0.0,0.09,0.33,83.46,0.15,3.31,8.56,35.93,0.18,0.0,94.0,0.0,0.0


## 合併

### 除息 × 配股／減資：同日重疊檢查
分別對 `ex_rights_adj_factor_df`（配股）與 `capital_reduction_only_df`（減資）檢查

In [36]:
# 檢查：同日是否同時有現金除息 ×（配股 / 減資）因子
cash_meta = (
    dividend_info
    .filter(pl.col("adj_factor").is_not_null())
    .select([
        "symbol",
        "date",
        pl.col("ex_div_ref_price").alias("cash_ref"),
        pl.col("adj_factor").alias("cash_adj"),
    ])
)

def _overlap_check(rights_df: pl.DataFrame, label: str):
    overlap = rights_df.join(cash_meta, on=["symbol", "date"], how="inner")
    same = overlap.filter(pl.col("ref_price") == pl.col("cash_ref"))
    diff = overlap.filter(pl.col("ref_price") != pl.col("cash_ref"))
    print(f"\n[{label}] 同日重疊總筆數: {overlap.height}")
    print(f"  參考價相同: {same.height}")
    print(f"  參考價不同: {diff.height}")
    if same.height:
        print(f"=== {label} 同日且參考價相同（應只保留一側）===")
        display(
            same.select(
                ["symbol", "date", "ref_price", "cash_ref", "adj_factor", "cash_adj"]
            ).sort(["symbol", "date"])
        )
    if diff.height:
        print(f"=== {label} 同日但參考價不同（不可自動相乘）===")
        display(
            diff.select(
                ["symbol", "date", "ref_price", "cash_ref", "adj_factor", "cash_adj"]
            ).sort(["symbol", "date"])
        )
    return same, diff

same_ref, diff_ref = _overlap_check(ex_rights_adj_factor_df, "配股")
same_ref_capred, diff_ref_capred = _overlap_check(
    capital_reduction_only_df, "減資"
)



[配股] 同日重疊總筆數: 1640
  參考價相同: 1640
  參考價不同: 0
=== 配股 同日且參考價相同（應只保留一側）===


symbol,date,ref_price,cash_ref,adj_factor,cash_adj
str,date,f64,f64,f64,f64
"""1101""",2018-07-26,39.63,39.63,1.138027,1.138027
"""1101""",2019-08-13,37.98,37.98,1.157188,1.157188
"""1101""",2020-08-12,41.9,41.9,1.109785,1.109785
"""1101""",2022-07-21,36.91,36.91,1.127066,1.127066
"""1103""",2023-07-27,20.63,20.63,1.034901,1.034901
…,…,…,…,…,…
"""9946""",2020-11-19,16.7,16.7,1.140719,1.140719
"""9946""",2021-09-29,15.61,15.61,1.069827,1.069827
"""9950""",2015-08-26,16.52,16.52,1.101695,1.101695



[減資] 同日重疊總筆數: 0
  參考價相同: 0
  參考價不同: 0


### 合併配股／減資因子到 OHLCV
- `ex_rights_adj_factor_df`（配股）與 `capital_reduction_only_df`（減資）分開 join
- 同日且參考價相同：丟掉該側，保留現金除息
- 同日但參考價不同：raise


In [37]:
# 合併：OHLCV（已有現金除息）× 配股因子 × 減資因子
# 同日且參考價相同 → 丟掉該側（配股/減資），保留現金除息
# 同日但參考價不同 → raise，禁止自動相乘
if diff_ref.height or diff_ref_capred.height:
    raise ValueError(
        f"同日參考價不同：配股 {diff_ref.height} 筆、減資 {diff_ref_capred.height} 筆；"
        "請人工決定保留哪一側後再跑本格"
    )

def _rights_to_join(rights_df: pl.DataFrame, same: pl.DataFrame) -> pl.DataFrame:
    out = (
        rights_df
        .join(same.select(["symbol", "date"]), on=["symbol", "date"], how="anti")
        .select(["symbol", "date", "adj_factor"])
    )
    _dup = out.group_by(["symbol", "date"]).len().filter(pl.col("len") > 1)
    if _dup.height:
        display(_dup)
        raise ValueError(f"合併前仍有重複 (symbol, date)：{_dup.height}")
    return out

ex_rights_to_join = _rights_to_join(ex_rights_adj_factor_df, same_ref)
capred_to_join = _rights_to_join(capital_reduction_only_df, same_ref_capred)

final_df = (
    OHLCV_df
    .join(ex_rights_to_join, on=["symbol", "date"], how="left")
    .with_columns(pl.col("adj_factor_right").fill_null(1).alias("adj_stock"))
    .drop("adj_factor_right")
    .join(capred_to_join, on=["symbol", "date"], how="left")
    .with_columns(pl.col("adj_factor_right").fill_null(1).alias("adj_capred"))
    .drop("adj_factor_right")
    .with_columns(
        (pl.col("adj_factor") * pl.col("adj_stock") * pl.col("adj_capred")).alias(
            "adj_factor"
        )
    )
    .drop(["adj_stock", "adj_capred"])
)

print(
    f"合併完成：丟掉同日同參考價 配股 {same_ref.height} / 減資 {same_ref_capred.height}；"
    f"寫入 配股 {ex_rights_to_join.height} / 減資 {capred_to_join.height}"
)
final_df.head()


合併完成：丟掉同日同參考價 配股 1640 / 減資 0；寫入 配股 368 / 減資 500


symbol,date,open,high,low,close,volume,amount,return_day,turnover,shares_outstanding_thousand,market_value_million,last_bid,last_ask,log_return_day,market_value_pct,trading_value_pct,numtrades,pe_ratio_tse,pe_ratio_tej,pb_ratio_tse,pb_ratio_tej,limit_status,ps_ratio_tej,div_yield_tse,cash_div_yield,price_change,spread,next_open_ref,next_limit_up,next_limit_down,attention_stock,disposed_stock,full_delivery,market,untradable,avoid_early,adj_factor
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,i8,i8,f64
"""1259""",2015-06-01,65.1,65.1,65.1,65.1,1.0,65.0,-0.4587,0.0031,32390.0,2109.0,65.1,65.9,-0.4598,0.073,0.0,1.0,20.47,21.02,1.34,1.34,null,0.49,3.07,3.53,-0.3,0.0,65.1,71.6,58.6,null,null,null,"""OTC""",0,0,1.0
"""1264""",2015-06-01,226.0,226.0,219.5,222.0,74.0,16443.0,-1.7699,0.2416,30633.0,6800.0,222.0,224.0,-1.7858,0.237,0.067,62.0,18.06,19.52,4.04,4.42,null,1.88,3.6,4.5,-4.0,2.8761,222.0,244.0,200.0,null,null,null,"""OTC""",0,0,1.0
"""1336""",2015-06-01,10.7,10.7,10.5,10.55,36.0,378.0,0.4762,0.0574,62753.0,662.0,10.5,10.55,0.4751,0.023,0.002,13.0,null,null,0.44,0.44,null,0.34,0.76,0.0,0.05,1.9048,10.55,11.6,9.5,null,null,null,"""OTC""",0,0,1.0
"""1565""",2015-06-01,547.0,550.0,538.0,541.0,104.0,56789.0,-0.3683,0.2063,50417.0,27275.0,541.0,543.0,-0.369,0.949,0.233,103.0,20.06,20.05,5.89,5.89,null,5.08,4.16,4.41,-2.0,2.2099,541.0,595.0,487.0,null,null,null,"""OTC""",0,0,1.0
"""1569""",2015-06-01,16.4,16.6,16.15,16.2,97.0,1584.0,-1.2195,0.1034,93785.0,1519.0,16.2,16.35,-1.227,0.053,0.007,74.0,null,null,0.68,0.76,null,0.57,4.94,0.0,-0.2,2.7439,16.2,17.8,14.6,null,null,null,"""OTC""",0,0,1.0


In [38]:
# 可以同時用 Python 3.8+ polars 的 reduce/pipe 簡化多個 join
import functools
import polars as pl

# 欲合併的 dataframe 列表
dfs_to_join = [
    institutional_return_df,
    institutional_net_flow_df ,
    Credit_Transaction_df , 
    Day_Trading_Stats_df  , 
    Securities_Lending_df , 
    share_distribution_df , 
]

final_df = functools.reduce(
    lambda left, right: left.join(right, on=["symbol", "date"], how="left"),
    dfs_to_join,
    final_df
)
final_df.head()

symbol,date,open,high,low,close,volume,amount,return_day,turnover,shares_outstanding_thousand,market_value_million,last_bid,last_ask,log_return_day,market_value_pct,trading_value_pct,numtrades,pe_ratio_tse,pe_ratio_tej,pb_ratio_tse,pb_ratio_tej,limit_status,ps_ratio_tej,div_yield_tse,cash_div_yield,price_change,spread,next_open_ref,next_limit_up,next_limit_down,attention_stock,disposed_stock,full_delivery,market,untradable,avoid_early,…,short_to_margin_ratio,margin_change_ratio,short_change_ratio,margin_volume_pct,short_volume_pct,margin_maintenance_ratio,short_maintenance_ratio,account_maintenance_ratio,margin_buy_lot,margin_sell_lot,margin_cash_repay_lot,short_buy_lot,short_sell_lot,short_stock_repay_lot,short_last_cover_date,margin_cash_repay_ratio,short_stock_repay_ratio,day_trade_volume_mkt_pct,day_trade_margin_short_offset_pct,lend_exch_balance,lend_broker_balance,lend_exch_net,lend_broker_net,foreign_trade_share_pct,it_trade_share_pct,dealer_trade_share_pct,credit_trade_share_pct,cash_trade_share_pct,free_float_turnover_pct,foreign_holding_ex_dir_pct,director_holding_pct,director_pledge_pct,margin_short_offset_pct,margin_increase_lot,margin_decrease_lot,short_increase_lot,short_decrease_lot
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,i8,i8,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""1259""",2015-06-01,65.1,65.1,65.1,65.1,1.0,65.0,-0.4587,0.0031,32390.0,2109.0,65.1,65.9,-0.4598,0.073,0.0,1.0,20.47,21.02,1.34,1.34,null,0.49,3.07,3.53,-0.3,0.0,65.1,71.6,58.6,null,null,null,"""OTC""",0,0,…,0.0,-0.25,0.0,50.0,0.0,181.26,null,181.26,0.0,1.0,0.0,0.0,0.0,0.0,null,0.0,null,null,null,null,null,null,null,0.0,0.0,0.0,0.5,99.5,0.01,5.55,61.51,5.42,0.0,0.0,1.0,0.0,0.0
"""1264""",2015-06-01,226.0,226.0,219.5,222.0,74.0,16443.0,-1.7699,0.2416,30633.0,6800.0,222.0,224.0,-1.7858,0.237,0.067,62.0,18.06,19.52,4.04,4.42,null,1.88,3.6,4.5,-4.0,2.8761,222.0,244.0,200.0,null,null,null,"""OTC""",0,0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,4.05,0.0,95.95,0.32,0.44,24.52,0.0,null,0.0,0.0,0.0,0.0
"""1336""",2015-06-01,10.7,10.7,10.5,10.55,36.0,378.0,0.4762,0.0574,62753.0,662.0,10.5,10.55,0.4751,0.023,0.002,13.0,null,null,0.44,0.44,null,0.34,0.76,0.0,0.05,1.9048,10.55,11.6,9.5,null,null,null,"""OTC""",0,0,…,2.62,-0.03,0.0,26.39,0.0,153.08,201.04,154.93,9.0,10.0,0.0,0.0,0.0,0.0,null,0.0,0.0,null,null,null,null,null,null,0.0,0.0,0.0,0.26,99.74,0.07,0.55,22.37,0.0,0.0,0.0,1.0,0.0,0.0
"""1565""",2015-06-01,547.0,550.0,538.0,541.0,104.0,56789.0,-0.3683,0.2063,50417.0,27275.0,541.0,543.0,-0.369,0.949,0.233,103.0,20.06,20.05,5.89,5.89,null,5.08,4.16,4.41,-2.0,2.2099,541.0,595.0,487.0,null,null,null,"""OTC""",0,0,…,4.32,-1.01,0.0,16.35,0.0,200.15,190.65,199.4,13.0,21.0,1.0,0.0,0.0,0.0,null,0.001138,0.0,13.4,10.53,1390.0,130.0,0.0,0.0,26.92,1.44,2.88,10.74,58.01,0.23,60.37,9.64,0.0,10.58,0.0,9.0,0.0,0.0
"""1569""",2015-06-01,16.4,16.6,16.15,16.2,97.0,1584.0,-1.2195,0.1034,93785.0,1519.0,16.2,16.35,-1.227,0.053,0.007,74.0,null,null,0.68,0.76,null,0.57,4.94,0.0,-0.2,2.7439,16.2,17.8,14.6,null,null,null,"""OTC""",0,0,…,0.95,-0.26,71.43,17.01,10.31,156.96,195.98,157.53,10.0,23.0,0.0,0.0,20.0,0.0,null,0.0,0.0,null,null,311.0,66.0,0.0,0.0,0.0,0.0,0.0,3.37,96.63,0.13,0.49,21.32,0.0,3.09,0.0,13.0,20.0,0.0


### 減資停牌期間：從 final_df 移除不應存在的交易日
`capital_reduction_halt_df` 區間 `[start_halt, resume)`（含恢復日前一交易日）理論上該 symbol 不應有日線；若有則刪除。

In [39]:
# 減資停牌 [start_halt, resume)：理論上 OHLCV/final_df 不應有列 → 列出不合理樣本後刪除
_halts = (
    capital_reduction_halt_df
    .filter(pl.col("resume_trading_date").is_not_null())
    .select(["symbol", "start_halt", "resume_trading_date"])
    .unique()
    .sort(["symbol", "start_halt"])
)

# 區間內日曆日
_should_absent = (
    _halts
    .join(calendar_df.select(pl.col("date").alias("cal_date")), how="cross")
    .filter(
        (pl.col("cal_date") >= pl.col("start_halt"))
        & (pl.col("cal_date") < pl.col("resume_trading_date"))
    )
    .select(["symbol", "start_halt", "resume_trading_date", pl.col("cal_date").alias("date")])
)

# 不合理 = 區間內 OHLCV 仍有列的停牌段
_ohlcv_in_halt = (
    OHLCV_df.select(["symbol", "date"])
    .join(
        _should_absent.select(["symbol", "date", "start_halt", "resume_trading_date"]),
        on=["symbol", "date"],
        how="inner",
    )
)
_bad_halts = (
    _ohlcv_in_halt
    .group_by(["symbol", "start_halt", "resume_trading_date"])
    .agg(pl.len().alias("n_ohlcv_in_range"))
    .sort(["symbol", "start_halt"])
)
print(f"區間內仍有 OHLCV 列的停牌段：{_bad_halts.height}（顯示前 5 段）")
display(_bad_halts.head(5))

_show_halts = _bad_halts.head(5)
for row in _show_halts.iter_rows(named=True):
    sym, st, rs = row["symbol"], row["start_halt"], row["resume_trading_date"]
    print("=" * 60)
    print(f"symbol={sym}  halt=[{st}, {rs})")

    in_range = (
        OHLCV_df
        .filter(
            (pl.col("symbol") == sym)
            & (pl.col("date") >= st)
            & (pl.col("date") < rs)
        )
        .select(["symbol", "date", "close", "volume", "untradable"])
        .sort("date")
    )
    print(f"【刪除前】區間內 OHLCV 共 {in_range.height} 日：")
    display(in_range)
    n_vol0 = in_range.filter(
        pl.col("volume").is_null() | (pl.col("volume") == 0)
    ).height
    print(
        f"  volume==0 或 null：{n_vol0}/{in_range.height}；"
        f"volume>0：{in_range.height - n_vol0}"
    )
    if in_range.height - n_vol0:
        print("  !! 有成交量>0 的列：")
        display(in_range.filter(pl.col("volume").fill_null(0) > 0))

# 全體被刪列：確認 volume 是否皆為 0
_drop_keys = _should_absent.select(["symbol", "date"]).unique()
_deleted_ohlcv = (
    OHLCV_df
    .join(_drop_keys, on=["symbol", "date"], how="inner")
    .select(["symbol", "date", "close", "volume", "untradable"])
)
_vol_pos = _deleted_ohlcv.filter(pl.col("volume").fill_null(0) > 0)
_vol_zero = _deleted_ohlcv.filter(pl.col("volume").is_null() | (pl.col("volume") == 0))
print("\n=== 全部將刪除列的成交量檢查 ===")
print(f"OHLCV 中會被刪的列：{_deleted_ohlcv.height}")
print(f"  volume==0 或 null：{_vol_zero.height}")
print(f"  volume>0：{_vol_pos.height}")
if _vol_pos.height:
    print("volume>0 樣本（不該在停牌日有量）：")
    display(_vol_pos.sort(["symbol", "date"]).head(20))
else:
    print("ok：被刪日期的成交量皆為 0 或 null")

# 從 final_df 刪除
_before = final_df.height
final_df = final_df.join(_drop_keys, on=["symbol", "date"], how="anti")
print(f"\nfinal_df：{_before} → {final_df.height}（刪 {_before - final_df.height} 列）")

# 同樣前 5 段：刪除後附近還剩哪些日期
print("\n【刪除後】前 5 段附近還剩的日期：")
cals = calendar_df.sort("date")["date"].to_list()
for row in _show_halts.iter_rows(named=True):
    sym, st, rs = row["symbol"], row["start_halt"], row["resume_trading_date"]
    try:
        i_st = cals.index(st)
    except ValueError:
        i_st = next((i for i, d in enumerate(cals) if d >= st), 0)
    try:
        i_rs = cals.index(rs)
    except ValueError:
        i_rs = next((i for i, d in enumerate(cals) if d >= rs), len(cals) - 1)
    win_start = cals[max(0, i_st - 5)]
    win_end = cals[min(len(cals) - 1, i_rs + 5)]

    left = (
        final_df
        .filter(
            (pl.col("symbol") == sym)
            & (pl.col("date") >= win_start)
            & (pl.col("date") <= win_end)
        )
        .select(["symbol", "date", "close", "untradable", "adj_factor"])
        .sort("date")
    )
    still_in = left.filter((pl.col("date") >= st) & (pl.col("date") < rs))
    print("=" * 60)
    print(f"symbol={sym}  halt=[{st}, {rs})  視窗=[{win_start}, {win_end}]")
    print(f"刪除後視窗內剩餘 {left.height} 日；其中仍在停牌區間：{still_in.height}")
    display(left)


區間內仍有 OHLCV 列的停牌段：500（顯示前 5 段）


symbol,start_halt,resume_trading_date,n_ohlcv_in_range
str,date,date,u32
"""1109""",2015-10-16,2015-10-30,10
"""1109""",2017-10-19,2017-10-30,7
"""1109""",2018-10-11,2018-10-22,7
"""1236""",2023-09-27,2023-10-11,7
"""1315""",2020-10-15,2020-10-26,7


symbol=1109  halt=[2015-10-16, 2015-10-30)
【刪除前】區間內 OHLCV 共 10 日：


symbol,date,close,volume,untradable
str,date,f64,f64,i8
"""1109""",2015-10-16,10.45,0.0,1
"""1109""",2015-10-19,10.45,0.0,1
"""1109""",2015-10-20,10.45,0.0,1
"""1109""",2015-10-21,10.45,0.0,1
"""1109""",2015-10-22,10.45,0.0,1
"""1109""",2015-10-23,10.45,0.0,1
"""1109""",2015-10-26,10.45,0.0,1
"""1109""",2015-10-27,10.45,0.0,1
"""1109""",2015-10-28,10.45,0.0,1


  volume==0 或 null：10/10；volume>0：0
symbol=1109  halt=[2017-10-19, 2017-10-30)
【刪除前】區間內 OHLCV 共 7 日：


...

symbol,date,close,untradable,adj_factor
str,date,f64,i8,f64
"""1109""",2018-10-03,15.1,0,1.0
"""1109""",2018-10-04,15.1,0,1.0
"""1109""",2018-10-05,14.95,0,1.0
"""1109""",2018-10-08,14.85,0,1.0
"""1109""",2018-10-09,14.75,1,1.0
…,…,…,…,…
"""1109""",2018-10-23,13.35,0,1.0
"""1109""",2018-10-24,13.35,0,1.0
"""1109""",2018-10-25,12.8,0,1.0


symbol=1236  halt=[2023-09-27, 2023-10-11)  視窗=[2023-09-20, 2023-10-18]
刪除後視窗內剩餘 11 日；其中仍在停牌區間：0


symbol,date,close,untradable,adj_factor
str,date,f64,i8,f64
"""1236""",2023-09-20,21.0,0,1.0
"""1236""",2023-09-21,20.7,0,1.0
"""1236""",2023-09-22,20.8,0,1.0
"""1236""",2023-09-25,20.75,0,1.0
"""1236""",2023-09-26,20.6,1,1.0
…,…,…,…,…
"""1236""",2023-10-12,23.35,0,1.0
"""1236""",2023-10-13,23.5,0,1.0
"""1236""",2023-10-16,23.55,0,1.0


symbol=1315  halt=[2020-10-15, 2020-10-26)  視窗=[2020-10-07, 2020-11-02]
刪除後視窗內剩餘 11 日；其中仍在停牌區間：0


symbol,date,close,untradable,adj_factor
str,date,f64,i8,f64
"""1315""",2020-10-07,53.8,0,1.0
"""1315""",2020-10-08,53.5,0,1.0
"""1315""",2020-10-12,52.8,0,1.0
"""1315""",2020-10-13,52.3,0,1.0
"""1315""",2020-10-14,52.7,1,1.0
…,…,…,…,…
"""1315""",2020-10-27,68.3,0,1.0
"""1315""",2020-10-28,68.8,0,1.0
"""1315""",2020-10-29,68.2,0,1.0


In [43]:
import datetime

# 指定 symbol 和日期區間
my_symbol = "1109"  # 這裡換成你想查詢的 symbol [2015-10-16, 2015-10-30)
start_date = datetime.date(2015, 10, 1)  # 這裡換成你的起始日期
end_date = datetime.date(2015, 10, 16)     # 這裡換成你的結束日期

result = (
    final_df
    .filter(
        (pl.col("symbol") == my_symbol)
        & (pl.col("date") >= pl.lit(start_date))
        & (pl.col("date") <= pl.lit(end_date))
    )
    .sort("date")
).select(['symbol' , 'date' , 'avoid_early'])
print(f"symbol={my_symbol} 區間: [{start_date}, {end_date}] 共 {result.height} 筆")
display(result)

symbol=1109 區間: [2015-10-01, 2015-10-16] 共 10 筆


symbol,date,avoid_early
str,date,i8
"""1109""",2015-10-01,0
"""1109""",2015-10-02,0
"""1109""",2015-10-05,0
"""1109""",2015-10-06,0
"""1109""",2015-10-07,0
"""1109""",2015-10-08,0
"""1109""",2015-10-12,0
"""1109""",2015-10-13,0
"""1109""",2015-10-14,0


In [ ]:
# 用該股票前一交易日的收盤價填充 open 欄位缺失
filled_df = (
    final_df
    .with_columns(
        pl.col("close").shift(1).over("symbol").alias("_prev_close")
    )
    .with_columns(
        pl.when(pl.col("open").is_null())
        .then(pl.col("_prev_close"))
        .otherwise(pl.col("open"))
        .alias("open_filled")
    )
    .drop("_prev_close")
)

# 如需覆蓋原欄位，可這樣執行：
# final_df = filled_df.with_columns(pl.col("open_filled").alias("open")).drop("open_filled")

print("填充後 open 欄位範例（前 10 筆）：")
display(filled_df.select(["symbol", "date", "open", "open_filled"]).head(10))

In [49]:

skip_na_cols = ["symbol", "date" , 'high' , 'low' , 'close']  # 如果欄位在這個list就跳過不計算na率

for col in final_df.columns:
    if col in skip_na_cols:
        continue
    n_na = final_df.filter(pl.col(col).is_null()).height
    total = final_df.height
    ratio = n_na / total if total > 0 else 0
    print(f"Column '{col}': {n_na} NA values ({ratio:.2%})")

Column 'open': 319 NA values (0.01%)
Column 'volume': 0 NA values (0.00%)
Column 'amount': 0 NA values (0.00%)
Column 'return_day': 1 NA values (0.00%)
Column 'turnover': 0 NA values (0.00%)
Column 'shares_outstanding_thousand': 0 NA values (0.00%)
Column 'market_value_million': 0 NA values (0.00%)
Column 'last_bid': 22837 NA values (0.48%)
Column 'last_ask': 60701 NA values (1.29%)
Column 'log_return_day': 1 NA values (0.00%)
Column 'market_value_pct': 290912 NA values (6.18%)
Column 'trading_value_pct': 290912 NA values (6.18%)
Column 'numtrades': 0 NA values (0.00%)
Column 'pe_ratio_tse': 1135231 NA values (24.11%)
Column 'pe_ratio_tej': 1125555 NA values (23.90%)
Column 'pb_ratio_tse': 236556 NA values (5.02%)
Column 'pb_ratio_tej': 220 NA values (0.00%)
Column 'limit_status': 4634715 NA values (98.42%)
Column 'ps_ratio_tej': 2834 NA values (0.06%)
Column 'div_yield_tse': 246872 NA values (5.24%)
Column 'cash_div_yield': 13316 NA values (0.28%)
Column 'price_change': 862 NA values 

In [50]:
# 指定要檢查 NA 的欄位
col_to_check = "open"  # 這裡可以換成你要檢查的欄位名稱

na_rows = (
    final_df
    .filter(pl.col(col_to_check).is_null())
    .select(["symbol", "date", col_to_check])
)

print(f"下列為欄位 '{col_to_check}' 為 NA 的資料，共 {na_rows.height} 筆：")
display(na_rows)

下列為欄位 'open' 為 NA 的資料，共 319 筆：


symbol,date,open
str,date,f64
"""6530""",2015-06-10,null
"""6542""",2015-06-25,null
"""6535""",2015-07-16,null
"""6546""",2015-07-28,null
"""6548""",2015-07-31,null
…,…,…
"""4178""",2024-12-05,null
"""7818""",2025-02-07,null
"""7822""",2025-03-27,null
